**Nikhil Sunder**  
ICSRI Research Fellow, University of Miami | Open-Source Developer  
Herbert Business School, University of Miami  
B.S.B.A. Quantitative Economics & Finance; Minor in Mathematics  

**Correspondence:** nss106@miami.edu  
**Research profiles:** [GitHub](https://github.com/nikhilxsunder) · [ORCID](https://orcid.org/0009-0007-3323-1760) · [LinkedIn](https://www.linkedin.com/in/nikhil-sunder/) · [Zenodo](https://zenodo.org/search?q=metadata.creators.person_or_org.name:%22Sunder,+Nikhil%22)  
**Software:** [PyPI](https://pypi.org/user/nikhil.sunder/) · [Anaconda](https://anaconda.org/nikhil.sunder/)

# 10. Reinforcement learning: tabular Q-learning

Watkins's Q-learning on a discretised state and action grid, in each economy that passed the
transmission check of notebook `07`.

**Inputs:** the artifacts of notebooks `00` to `07`. **Outputs:** `artifacts/rl/qlearning/*.pt`.

In [ ]:
import pandas as pd

import autofed as af
from autofed import plots, rl

nb = af.NotebookSession(
    "10_rl_qlearning", number=10, title="Reinforcement learning: tabular Q-learning"
)

lab = rl.PolicyLab.load(
    nb.workspace.artifacts
)  # economies of notebooks 01-06, settings of notebook 07
cfg = lab.config
OBS_SPECS = ["x1", "x2"]
LOSS_COLUMNS = {
    "dev2 pi": r"$\Delta^2(\pi^*, \pi_t)$",
    "dev2 y": r"$\Delta^2(y^*, y_t)$",
    "dev2 di": r"$\Delta^2(\Delta i_t)$",
    "penalty": "Penalty",
}
PATH_NOTE = (
    "Simulated from the first two estimation-window quarters under freshly drawn shocks; every"
    " policy faces the same shocks within an economy. Not a historical counterfactual: the Actual"
    " line shares only its first two quarters with the simulations."
)


def slug(name: str) -> str:
    return name.lower().replace("-", "_").replace(" ", "_")


print(
    "Economies (passed the transmission check of notebook 07):",
    ", ".join(lab.env_names),
    "| episodes per run:",
    cfg.q_episodes,
    "| seed:",
    cfg.seed,
)

## The algorithm

TorchRL ships deep Q-learning (`DQNLoss`) but no tabular agent, and a table is the point of this
section: Wang, Feinstein & Chen (2026) report that on similar single-central-bank monetary MDPs tabular
Q-learning outperforms DQN and Bayesian variants. We implement Watkins's (1989) algorithm directly on a
PyTorch tensor (`autofed.rl.TabularQPolicy`, `train_q_cell`):

- **State.** $(y_t, \pi_t)$, the no-lag specification $x_t^{(1)}$, binned by interior edges that are fine
  around the targets and the stopping band and coarse in the tails (`Y_EDGES`, `PI_EDGES`); observations
  beyond the outer edges fall in the outer bins. The table does not read $i_{t-1}$.
- **Actions.** Rates from $0$ to $i_{\max}$ in steps of $0.5$ percentage points, the same range as the
  continuous agents.
- **Update.** $\varepsilon$-greedy exploration with per-episode decay and a visit-count step size
  $\alpha_n = \alpha_0 / n^{0.6}$; the target bootstraps through truncation at $T$ and not through
  termination in the band, like the TD(0) targets of DDPG and SAC.
- **Greedy policy.** Every reward is non-positive, so an action the agent never tried would keep its
  initial value of zero and beat every tried one; the greedy choice is therefore taken over visited
  actions only, and a state the agent never visited falls back to the TR93 rule.

The agent interacts with the same TorchRL environment as DDPG and SAC, stepping it with
`step_and_maybe_reset`, and is reported with the same two statistics (steady-state reward at the
deterministic fixed point, evaluation reward over stochastic episodes).

In [ ]:
cells = [rl.train_q_cell(lab, env_name) for env_name in lab.env_names]
saved = rl.save_cells(cells, nb.artifacts("rl", "qlearning"))
print(
    f"Trained and saved {len(saved)} Q-learning cells to"
    f" {saved[0].parent.relative_to(nb.workspace.root)}"
)

policy = cells[0].policy
trained = pd.DataFrame({
    c.env_name: {
        "SS reward": c.steady_state_reward,
        "Eval reward": c.eval_reward,
        "Mean return, last 1000 episodes": float(pd.Series(c.training_rewards[-1000:]).mean()),
        "Episodes": len(c.training_rewards),
        "Visited state-action pairs": int((c.policy.N > 0).sum()),
    }
    for c in cells
}).T.astype({"Episodes": int, "Visited state-action pairs": int})
trained.index.name = "Economy"
nb.table(
    trained,
    "trained_cells",
    "Q-learning cells",
    float_format="{:.2f}",
    note=(
        f"State grid {policy.bins[0]} x {policy.bins[1]} over (y, pi), interior edges"
        f" {tuple(policy.y_edges.tolist())} and {tuple(policy.pi_edges.tolist())};"
        f" {policy.action_grid.numel()} rates from {policy.action_grid[0]:g} to"
        f" {policy.action_grid[-1]:g} percent. The table has {policy.Q.numel()} entries. SS reward:"
        " reward at the deterministic closed-loop steady state of the greedy policy; Eval reward:"
        f" mean per-step reward over {cfg.eval_episodes} stochastic episodes."
    ),
)

## Effective Taylor coefficients

Q-learning's discretised policy is non-linear and non-differentiable; the OLS $R^2$ measures how well a
Taylor-rule form fits the discretised reaction function over the grid. The greedy policy is evaluated
over visited actions, with the TR93 fallback in unvisited states.

In [ ]:
nb.table(
    rl.coefficient_table(cells),
    "taylor_coefficients",
    "Effective Taylor coefficients of the Q-learning policies",
    note=(
        "OLS of the greedy policy rate on a (pi, y) grid, in the level form i = alpha0 +"
        " beta_pi pi + beta_y y of Hinterlang & Tänzer (2021, Table 4). R2 measures how well a"
        " linear rule describes the policy."
    ),
)

## Training curves, reaction functions and simulated paths

In [ ]:
fig = plots.training_curves({c.env_name: c.training_rewards for c in cells}, window=1000)
nb.figure(
    fig,
    "training_curves",
    "Q-learning training: episode returns in each economy",
    note="Undiscounted episode return, 1000-episode rolling mean.",
)

In [ ]:
fig = plots.policy_surface_grid(
    {c.env_name: rl.policy_surface(c) for c in cells},
    rate_max=cfg.action_high,
    pi_star=cfg.pi_star,
    style="mesh",
)
nb.figure(
    fig,
    "reaction_functions",
    "Greedy Q-learning reaction functions in each economy",
    note=(
        "Policy rate of the value-maximising visited action in each state cell, drawn without"
        " interpolation. In state cells the agent never visited the TR93 rule is applied."
    ),
)

In [ ]:
paths_by_economy = {
    env_name: lab.simulate_policies(env_name, cells) for env_name in lab.env_names
}
for env_name, paths in paths_by_economy.items():
    fig = plots.policy_paths(paths, pi_star=cfg.pi_star, relative_time=True)
    nb.figure(
        fig,
        f"simulated_paths_{slug(env_name)}",
        f"Q-learning policies and the reference rules in the {env_name} economy",
        note=PATH_NOTE,
    )

In [ ]:
loss_table = lab.loss_table(paths_by_economy).rename(columns=LOSS_COLUMNS)
nb.table(
    loss_table,
    "loss_decomposition",
    "Loss decomposition of the simulated paths: Q-learning and the reference rules",
    note=(
        "Mean squared deviations over the simulated quarters; Loss weights them as the reward"
        " does. Penalty is the mean footnote-13 term, so Loss + Penalty is minus the mean reward."
        " The Actual row is the same statistic on the realised data and is a reference level, not"
        " a like-for-like comparison."
    ),
)

# References

- Hinterlang, N., & Tänzer, A. (2021). Optimal monetary policy using reinforcement learning. *Deutsche Bundesbank Discussion Paper No. 51/2021*.
- Mnih, V., Kavukcuoglu, K., Silver, D., Rusu, A. A., Veness, J., Bellemare, M. G., Graves, A., Riedmiller, M., Fidjeland, A. K., Ostrovski, G., Petersen, S., Beattie, C., Sadik, A., Antonoglou, I., King, H., Kumaran, D., Wierstra, D., Legg, S., & Hassabis, D. (2015). Human-level control through deep reinforcement learning. *Nature*, 518(7540), 529-533.
- Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.
- Taylor, J. B. (1993). Discretion versus policy rules in practice. *Carnegie-Rochester Conference Series on Public Policy*, 39, 195-214.
- Wang, T., Feinstein, K., & Chen, S. (2026). Reinforcement learning for monetary policy under macroeconomic uncertainty: Analyzing tabular and function approximation methods. *arXiv preprint arXiv:2512.17929*.
- Watkins, C. J. C. H. (1989). *Learning from delayed rewards* (Doctoral dissertation). King's College, University of Cambridge.
- Watkins, C. J. C. H., & Dayan, P. (1992). Q-learning. *Machine Learning*, 8(3-4), 279-292.


# Export

The last cell converts this notebook to PDF in the shared `documents/` directory. A kernel can only
convert the copy saved on disk, so the cell waits briefly for the editor to save; if the file is still
behind, it writes nothing and asks you to save and run it again.

In [ ]:
nb.export_pdf()

/var/folders/cj/p5jdwnb923752l7zjc_69j5w0000gn/T/ipykernel_80437/281924391.py:1: UserWarning: 10_rl_qlearning.ipynb on disk does not yet contain this run's outputs, so no PDF was written. Save the notebook and run this cell again.
  nb.export_pdf()
